In [32]:
!pip install -qq databricks databricks-sdk

In [1]:
!pip install -qq 'crewai[tools]'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.0/559.0 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 5.4 MB/s eta 0:

In [42]:
import os
from google.colab import userdata
os.environ['AZURE_API_KEY'] = userdata.get('AZURE_API_KEY')
os.environ['AZURE_OPENAI_API_KEY'] = userdata.get('AZURE_API_KEY')
os.environ['AZURE_API_VERSION'] = userdata.get('AZURE_API_VERSION')
os.environ['AZURE_API_BASE'] = userdata.get('AZURE_BASE_URL')
os.environ['AZURE_OPENAI_ENDPOINT'] = userdata.get('AZURE_BASE_URL')
os.environ['SERPER_API_KEY'] = userdata.get('SERPER_DEV_API_KEY')

In [35]:
from crewai import LLM
from pydantic import BaseModel, Field
class Section(BaseModel):
    answer: str = Field(description="The answer to the question.")
class SuperSection(Section):
    data: Section = Field(description="The answer to the question.")
llm = LLM(f"azure/gpt-4o",
        #  response_format=Section
    # response_format={"type": "json_object"}
          )
llm.call(f"What is the capital of France?, response following JSON schema:{SuperSection.model_json_schema()}")

 
[2025-03-14 22:30:56][🤖 LLM CALL STARTED]: 2025-03-14 22:30:56.223224
 
[2025-03-14 22:30:57][✅ LLM CALL COMPLETED]: 2025-03-14 22:30:57.109499


'```json\n{\n  "answer": "The capital of France is Paris.",\n  "data": {\n    "answer": "Paris"\n  }\n}\n```'

In [36]:
SuperSection.model_json_schema()

{'$defs': {'Section': {'properties': {'answer': {'description': 'The answer to the question.',
     'title': 'Answer',
     'type': 'string'}},
   'required': ['answer'],
   'title': 'Section',
   'type': 'object'}},
 'properties': {'answer': {'description': 'The answer to the question.',
   'title': 'Answer',
   'type': 'string'},
  'data': {'$ref': '#/$defs/Section',
   'description': 'The answer to the question.'}},
 'required': ['answer', 'data'],
 'title': 'SuperSection',
 'type': 'object'}

In [37]:
from crewai import Agent, Task
from crewai import Crew, Process

# Create a researcher agent
researcher = Agent(
  role='Senior Researcher',
  goal='Discover groundbreaking technologies',
  verbose=True,
  llm=llm,
  backstory='A curious mind fascinated by cutting-edge innovation and the potential to change the world, you know everything about tech.'
)

# Task for the researcher
research_task = Task(
  description='Identify the next big trend in AI',
  expected_output='5 paragraphs on the next big AI trend in different message',
  agent=researcher  # Assigning the task to the researcher
)


# Instantiate your crew
tech_crew = Crew(
  agents=[researcher],
  tasks=[research_task],
  verbose=True,
  planning= True,
  planning_llm=llm,
  process=Process.sequential  # Tasks will be executed one after the other
)

# Begin the task execution
result = tech_crew.kickoff()
# result

 
[2025-03-14 22:31:28][🚀 CREW 'CREW' STARTED, 711C165E-B008-4D55-9F74-C89C7E8F3819]: 2025-03-14 22:31:28.737248
 
[2025-03-14 22:31:28][INFO]: Planning the crew execution
 
[2025-03-14 22:31:28][📋 TASK STARTED: BASED ON THESE TASKS SUMMARY: 
                TASK NUMBER 1 - IDENTIFY THE NEXT BIG TREND IN AI
                "TASK_DESCRIPTION": IDENTIFY THE NEXT BIG TREND IN AI
                "TASK_EXPECTED_OUTPUT": 5 PARAGRAPHS ON THE NEXT BIG AI TREND IN DIFFERENT MESSAGE
                "AGENT": SENIOR RESEARCHER
                "AGENT_GOAL": DISCOVER GROUNDBREAKING TECHNOLOGIES
                "TASK_TOOLS": []
                "AGENT_TOOLS": "AGENT HAS NO TOOLS" 
 CREATE THE MOST DESCRIPTIVE PLAN BASED ON THE TASKS DESCRIPTIONS, TOOLS AVAILABLE, AND AGENTS' GOALS FOR THEM TO EXECUTE THEIR GOALS WITH PERFECTION.]: 2025-03-14 22:31:28.752782
 
[2025-03-14 22:31:28][🤖 AGENT 'TASK EXECUTION PLANNER' STARTED TASK]: 2025-03-14 22:31:28.753619
 
[2025-03-14 22:31:28][🤖 LLM CALL STARTED]: 20

In [43]:
from crewai_tools import WebsiteSearchTool, SerperDevTool

# Instantiate Web Search Tool
web_search_tool = WebsiteSearchTool(
    config=dict(
        llm=dict(
            provider="azure_openai", # or google, openai, anthropic, llama2, ...
            config=dict(
                model="gpt-4o",
                deployment_name="gpt-4o",
            ),
        ),
        embedder=dict(
            provider="azure_openai", # or openai, ollama, ...
            config=dict(
                model="cwstxtemb3small",
                deployment_name="cwstxtemb3small",
                # deployment_name="text-embedding-3-small",
                # deployment_name="text-embedding-3-small",
            ),
        ),
    ),
)
search_tool = SerperDevTool(search_type='news')

In [44]:
web_search_tool._run(search_query='latest trends in AI', website='www.techcrunch.com')
# web_search_tool.run('latest trends in AI' )

Inserting batches in chromadb: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]


'Relevant Content:\nwww.techcrunch.com'

In [45]:
search_tool._run(search_query='AI in peruvian startups', )

{'searchParameters': {'q': 'AI in peruvian startups',
  'type': 'news',
  'num': 10,
  'engine': 'google'},
 'news': [{'title': '10 Top Construction AI Startups and Companies to Watch in 2025',
   'link': 'https://www.startus-insights.com/innovators-guide/construction-ai-startups/',
   'snippet': 'Construction AI startups are advancing the industry by integrating artificial intelligence into project management, defect detection,...',
   'date': '3 weeks ago',
   'source': 'StartUs Insights',
   'imageUrl': 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRH6IOV96Y9bbdN8vDcS-x71d202q4T1JYNzfKG7E4d24XtCTctEDoTID3jTg&s'},
  {'title': "Artificio Tackles Autonomous Driving's Toughest Challenge in Lima •",
   'link': 'https://contxto.com/en/peru/artificio-tackles-autonomous-drivings-toughest-challenge-in-lima/',
   'snippet': "Artificio, founded by Arturo Deza, a former MIT postdoctoral researcher and current professor at Lima's University of Engineering and...",
   'date': '9 months a

In [46]:
from crewai import Agent

# Create Agents
researcher = Agent(
    role='Market Research Analyst',
    goal='Provide up-to-date market analysis of the AI industry',
    backstory='An expert analyst with a keen eye for market trends.',
    tools=[search_tool,web_search_tool],  # Only uses web search tool
    verbose=True,
    llm=llm
)

In [47]:
writer = Agent(
    role='Content Writer',
    goal='Craft engaging blog posts about the AI industry',
    backstory='A skilled writer with a passion for technology.',
    tools=[],  # No tools needed; writes based on research summary
    verbose=True,
    llm=llm
)

In [48]:
from crewai import Task, Crew

# Define Tasks
research = Task(
    description='Search the web for the latest AI trends and provide a summarized report in 2025.',
    expected_output='A summary of the top 3 trending developments in AI with insights on their impact.',
    agent=researcher
)

In [49]:
write = Task(
    description='Write an engaging blog post about the AI industry based on the research analyst’s summary.',
    expected_output='A well-structured, 4-paragraph blog post in markdown format with simple, engaging content.',
    agent=writer,
    output_file='blog-posts/new_post.md'  # Saves blog post in 'blog-posts' directory
)

In [50]:
# Assemble the Crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[research, write],
    verbose=True,
    planning=True,  # Enable AI planning feature
    planning_llm=llm
)

# Execute the Tasks
result = crew.kickoff()

 
[2025-03-14 22:38:57][🚀 CREW 'CREW' STARTED, B753512A-C9D0-493C-8A23-D9667418849C]: 2025-03-14 22:38:57.938148
 
[2025-03-14 22:38:57][INFO]: Planning the crew execution
 
[2025-03-14 22:38:57][📋 TASK STARTED: BASED ON THESE TASKS SUMMARY: 
                TASK NUMBER 1 - SEARCH THE WEB FOR THE LATEST AI TRENDS AND PROVIDE A SUMMARIZED REPORT IN 2025.
                "TASK_DESCRIPTION": SEARCH THE WEB FOR THE LATEST AI TRENDS AND PROVIDE A SUMMARIZED REPORT IN 2025.
                "TASK_EXPECTED_OUTPUT": A SUMMARY OF THE TOP 3 TRENDING DEVELOPMENTS IN AI WITH INSIGHTS ON THEIR IMPACT.
                "AGENT": MARKET RESEARCH ANALYST
                "AGENT_GOAL": PROVIDE UP-TO-DATE MARKET ANALYSIS OF THE AI INDUSTRY
                "TASK_TOOLS": [SERPERDEVTOOL(NAME='SEARCH THE INTERNET WITH SERPER', DESCRIPTION="TOOL NAME: SEARCH THE INTERNET WITH SERPER\NTOOL ARGUMENTS: {'SEARCH_QUERY': {'DESCRIPTION': 'MANDATORY SEARCH QUERY YOU WANT TO USE TO SEARCH THE INTERNET', 'TYPE': 'STR'}}\

Inserting batches in chromadb: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]


 
[2025-03-14 22:39:13][✅ TOOL USAGE FINISHED: 'SEARCH IN A SPECIFIC WEBSITE']: 2025-03-14 22:39:13.810869


# Agent: Market Research Analyst
## Using tool: Search in a specific website
## Tool Input: 
"{\"search_query\": \"AI trends 2025\", \"website\": \"https://www.nvidia.com\"}"
## Tool Output: 
Relevant Content:
Revolutionizing Manufacturing and Supply Chain with Digital Twins In today’s fast-paced consumer packaged goods (CPG) industry, PepsiCo is leading the way in digital transformation by harnessing. View Session Pioneering the Future of Data Platforms for the Era of Generative AI In the era of generative AI, retrieval augmented generation (RAG) is emerging as a vital tool, enabling enterprises to capitalize on their. View Session Previous Next s1 s2 s3 s4 s5 s6 s7 s8 s9 s10 s11 Artificial Intelligence AI is powering change in every industry. From generative AI and speech recognition to medical imaging and improved supply chain management, AI is providing enterprises that comp

In [51]:
from IPython.display import display, Markdown
display(Markdown(result.raw))

# The AI Revolution: Top Trends Shaping the Future

Artificial intelligence (AI) has been at the forefront of technological innovation, driving unprecedented changes across industries. As we look toward 2025, several emerging trends are poised to redefine the AI landscape, promising to accelerate growth, introduce new job opportunities, and solve complex challenges. Understanding these developments is crucial for anyone involved in the tech industry to fully leverage AI's potential. Let's dive into the top three AI trends that are set to shape our future.

## Generative AI: Redefining Creativity and Innovation

One of the most exciting advancements in AI is the rise of generative AI. This innovative technology enables machines to create new content, from text and images to music and even entire virtual environments. By leveraging advanced algorithms and deep learning models, generative AI can produce high-quality, human-like outputs. The potential impact of this trend is vast, spanning various industries such as entertainment, marketing, and design. For instance, generative AI can help marketers create personalized ad campaigns at scale, while artists can use it to collaborate with AI tools to push the boundaries of creativity. As generative AI continues to evolve, it promises to unlock new levels of innovation and productivity, making it a cornerstone of future technological progress.

## AI-Powered Agents: Revolutionizing Interactions and Efficiency

The second major trend is the proliferation of AI-powered agents, which are intelligent software entities designed to perform specific tasks autonomously. These agents can handle tasks ranging from customer service inquiries to complex data analysis, greatly enhancing efficiency and user experience. Unique aspects of AI-powered agents include their ability to learn and adapt over time, improving their performance with every interaction. Real-world applications are already evident, with virtual assistants like Siri and Alexa becoming household names. Businesses are also deploying AI agents for tasks such as automating customer support, leading to faster resolution times and improved customer satisfaction. The widespread adoption of AI-powered agents holds significant promise for increasing operational efficiency across various sectors.

## AI in Healthcare: Transformative Changes and Challenges

The third trend, AI in healthcare, is set to revolutionize the medical field by enhancing diagnostic accuracy, personalizing treatment plans, and improving patient outcomes. AI systems can analyze vast amounts of medical data to identify patterns that might be missed by human doctors, leading to earlier detection of diseases and more effective treatments. For example, AI algorithms are being used to predict patient deterioration in intensive care units, enabling timely interventions and potentially saving lives. However, the integration of AI in healthcare also presents challenges, including data privacy concerns and the need for continuous updates to AI models to maintain accuracy. Despite these challenges, the transformative potential of AI in healthcare is undeniable, promising to make healthcare more efficient and accessible.

## Conclusion

As we approach 2025, the trends of generative AI, AI-powered agents, and AI in healthcare are set to drive significant advancements in technology and society. These developments will not only spur industry growth and create new opportunities but also address critical issues and improve the quality of life. Staying updated with these trends is essential for harnessing the full potential of AI, making it an indispensable tool in our rapidly evolving world. By embracing these changes, we can look forward to a future where AI plays a pivotal role in shaping a better, more innovative society.

Ensure you stay informed and ready to leverage the latest AI advancements—because the future is happening now, and AI is leading the way.